# Il modello non vede lettere. Vede simboli, e se li e' scelti da solo

*Anatomia di un LLM* · capitolo 4, «Dal testo ai numeri» · nel libro: §4.1 «Il modello non vede lettere» · script `02-01`, `scripts/02_01_bpe_da_zero.py`

**Dove siamo.** Primo notebook del capitolo che porta il testo dentro il modello. Qui si costruisce da zero il BPE, il tokenizzatore dei modelli commerciali: il nostro userà i caratteri, ma vedere come si fondono i pezzi spiega metà delle stranezze del notebook successivo.

Prima di arrivare al modello, il testo va ridotto a numeri. Il modo usato
ovunque non spezza per parole ne' per lettere: parte dalle lettere e poi fonde
ripetutamente la coppia piu' frequente, finche' il vocabolario e' pieno. Si
chiama BPE, e in questa pagina lo costruiamo da zero.

Si esegue con `python3 scripts/02_01_bpe_da_zero.py`
e dura circa 15 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/02_01_bpe_da_zero.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [ ]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
from collections import Counter

from llm_anatomy.data import carica_testo
from llm_anatomy.utils import numero, prepara, tabella, titolo, verifica

prepara()

# Un corpus ridotto: l'algoritmo e' quadratico nel numero di merge e qui
# interessa vederlo lavorare, non produrre un tokenizer da usare davvero.
testo = carica_testo()[:200_000]
N_MERGE = 335

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


### Si parte dalle lettere

> **Nel libro, §4.1 «Il modello non vede lettere».** In quali unità conviene spezzare il testo: lettere, parole, o qualcosa in mezzo?

**Cosa guardare.** 82 simboli e 200.000 token: all'inizio ogni carattere è un simbolo.

In [ ]:
titolo("Si parte dalle lettere")
simboli = sorted(set(testo))
vocabolario = {c: i for i, c in enumerate(simboli)}
sequenza = [vocabolario[c] for c in testo]
print(f"  corpus:      {numero(len(testo))} caratteri")
print(f"  vocabolario: {len(vocabolario)} simboli")
print(f"  sequenza:    {numero(len(sequenza))} token")


def conta_coppie(seq: list[int]) -> Counter:
    """Conta ogni coppia di token adiacenti: l'unica statistica che serve."""
    return Counter(zip(seq, seq[1:]))


def fondi(seq: list[int], coppia: tuple[int, int], nuovo: int) -> list[int]:
    """Sostituisce ogni occorrenza della coppia con un solo token nuovo."""
    fuori, i = [], 0
    a, b = coppia
    while i < len(seq):
        if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
            fuori.append(nuovo)
            i += 2
        else:
            fuori.append(seq[i])
            i += 1
    return fuori


Si parte dalle lettere
----------------------
  corpus:      200.000 caratteri
  vocabolario: 82 simboli
  sequenza:    200.000 token


### Le prime dieci fusioni

**Cosa guardare.** Le prime fusioni sono vocale più spazio, virgola più spazio: quello che capita più spesso, senza significato.

In [ ]:
titolo("Le prime dieci fusioni")
inverso = {i: c for c, i in vocabolario.items()}
merge = []          # le fusioni, in ordine: ((a, b), nuovo simbolo)
righe = []
for passo in range(N_MERGE):
    conteggi = conta_coppie(sequenza)
    if not conteggi:
        break
    coppia, frequenza = conteggi.most_common(1)[0]
    nuovo = len(inverso)
    testo_nuovo = inverso[coppia[0]] + inverso[coppia[1]]
    inverso[nuovo] = testo_nuovo
    merge.append((coppia, nuovo))
    prima = len(sequenza)
    sequenza = fondi(sequenza, coppia, nuovo)
    if passo < 10:
        righe.append([
            passo + 1,
            f"{inverso[coppia[0]]!r} + {inverso[coppia[1]]!r}",
            repr(testo_nuovo),
            numero(frequenza),
            numero(prima - len(sequenza)),
        ])
tabella(["merge", "coppia", "nuovo simbolo", "occorrenze",
         "token risparmiati"],
        righe)
print("  i primi merge sono sequenze frequentissime e senza significato:")
print("  vocale piu' spazio, virgola piu' spazio, sillabe")

# Quanti dei merge appresi scavalcano un confine di parola: e' esattamente cio'
# che la pre-tokenizzazione di 02-02 rende impossibile. Lo splitter a regex
# ammette lo spazio solo in testa a un pezzo, e le sequenze di soli spazi come
# pezzo a se': tutto il resto e' un merge che unisce due parole diverse.
def a_cavallo(t: str) -> bool:
    resto = t.lstrip(" \n")           # l'eventuale spazio iniziale e' lecito
    return any(c in " \n" for c in resto)


scavalcano = [t for i, t in inverso.items()
              if i >= len(vocabolario) and a_cavallo(t)]
print(f"  merge che scavalcano un confine di parola: {len(scavalcano)}"
      f" su {N_MERGE}")
print(f"  i primi: {scavalcano[:6]}")
print("  in un tokenizer reale non esisterebbero: lo splitter spezza il testo")
print("  prima dei merge e nessuna fusione attraversa i suoi confini")


Le prime dieci fusioni
----------------------


merge     coppia  nuovo simbolo  occorrenze  token risparmiati
-----  ---------  -------------  ----------  -----------------
1      'e' + ' '           'e '       6.409              6.409
2      'o' + ' '           'o '       5.110              5.110
3      'a' + ' '           'a '       4.821              4.821
4      'i' + ' '           'i '       4.422              4.422
5      ',' + ' '           ', '       2.386              2.386
6      'a' + 't'           'at'       2.380              2.380
7      'e' + 'r'           'er'       2.228              2.228
8      'c' + 'o'           'co'       2.159              2.159
9      'e' + 'l'           'el'       2.008              2.008
10     'i' + 'n'           'in'       1.954              1.954
  i primi merge sono sequenze frequentissime e senza significato:
  vocale piu' spazio, virgola piu' spazio, sillabe
  merge che scavalcano un confine di parola: 129 su 335
  i primi: ['e ', 'o ', 'a ', 'i ', ', ', 'di ']
  in un tokenizer real

### Più simboli, sequenze più corte

**Cosa guardare.** La colonna `caratteri per token`: più fusioni, sequenze più corte.

**Prova a cambiare:** porta `N_MERGE` da 335 a 1000 (in cima al notebook).

<details><summary>Che cosa succede</summary>

L'ultima riga diventa 1000 fusioni, 1082 simboli, 3,28 caratteri per token: la sequenza continua ad accorciarsi, ma ogni fusione guadagna meno della precedente. La cella ci mette circa un minuto.

</details>

In [ ]:
titolo("Piu' simboli, sequenze piu' corte")
righe = []
for soglia in (0, 50, 150, N_MERGE):
    # Ricostruzione da capo fino al merge indicato: costa, ma rende visibile
    # la curva senza doversi fidare di un conteggio incrementale.
    s = [vocabolario[c] for c in testo]
    for coppia, nuovo in merge[:soglia]:
        s = fondi(s, coppia, nuovo)
    righe.append([
        soglia,
        len(vocabolario) + soglia,
        numero(len(s)),
        numero(len(testo) / len(s), 2),
    ])
tabella(["merge", "vocabolario", "token", "caratteri per token"], righe)


Piu' simboli, sequenze piu' corte
---------------------------------


merge  vocabolario    token  caratteri per token
-----  -----------  -------  -------------------
0               82  200.000                 1,00
50             132  129.757                 1,54
150            232  100.838                 1,98
335            417   82.083                 2,44


### Che cosa è diventato un simbolo solo

**Cosa guardare.** `'al fuoco '`, `'grammi '`, `'parmigi'`: il vocabolario ha imparato il ricettario da solo.

In [ ]:
titolo("Che cosa e' diventato un simbolo solo")
lunghi = sorted(((len(t), i, t) for i, t in inverso.items()
                 if i >= len(vocabolario)),
                reverse=True)[:12]
print("  i dodici simboli piu' lunghi del vocabolario finale:")
for lung, idx, t in lunghi:
    print(f"    id {idx:4d}  {t!r}")


Che cosa e' diventato un simbolo solo
-------------------------------------
  i dodici simboli piu' lunghi del vocabolario finale:
    id  412  'o, grammi '
    id  346  'al fuoco '
    id  388  'questa '
    id  382  'questo '
    id  370  'con un '
    id  350  'parmigi'
    id  347  'compost'
    id  340  'quanto '
    id  334  'aggiung'
    id  295  'minestr'
    id  253  'quando '
    id  184  'grammi '


### Avanti e indietro, senza perdere niente

*Questa parte non è stampata nel libro.*

**Cosa guardare.** La verifica: decodificare la sequenza fusa restituisce il testo identico.

In [ ]:
titolo("Avanti e indietro, senza perdere niente")
def decodifica(seq: list[int]) -> str:
    return "".join(inverso[i] for i in seq)

verifica(decodifica(sequenza) == testo,
         "la sequenza fusa decodifica esattamente al corpus di partenza")
print(f"  compressione finale: {len(testo)} caratteri -> {len(sequenza)} token")
print(f"  rapporto: {len(testo) / len(sequenza):.2f} caratteri per token"
      .replace(".", ","))
print("  un BPE reale su miliardi di caratteri arriva a 4 circa: qui il")
print("  corpus e' piccolo e i merge disponibili si esauriscono prima")


Avanti e indietro, senza perdere niente
---------------------------------------
  [verifica superata] la sequenza fusa decodifica esattamente al corpus di
                      partenza
  compressione finale: 200000 caratteri -> 82083 token
  rapporto: 2,44 caratteri per token
  un BPE reale su miliardi di caratteri arriva a 4 circa: qui il
  corpus e' piccolo e i merge disponibili si esauriscono prima
